# Tokenizer


## Step 1 : creating tokens


In [10]:
with open('the-verdict.txt','r',encoding='utf-8') as f:
    raw_text=f.read()

In [2]:
print('Total number of character :', len(raw_text))
print(raw_text[:99])

Total number of character : 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


Our goal is to tokenize this 20479 char short story into individual words and special characters that we cam then turn into embedding for llm training   

In [3]:
import re
text="hello, world. this, is the first."
result=re.split(r'(\s)',text) # split wherever there is a space
print(result)

['hello,', ' ', 'world.', ' ', 'this,', ' ', 'is', ' ', 'the', ' ', 'first.']


the result is  list of individual words, whitespaces, and punctuation characters.

modify the regular expression splits on whitespaces(\s) and commas , periods[,.]:

In [4]:
result=re.split(r'([,.]|\s)',text)
print(result)

['hello', ',', '', ' ', 'world', '.', '', ' ', 'this', ',', '', ' ', 'is', ' ', 'the', ' ', 'first', '.', '']


small issue - list still has whitespace characters. we can remove redundant characters safely:

In [5]:
result=[item for item in result if item.strip()] # remove empty strings
print(result)

['hello', ',', 'world', '.', 'this', ',', 'is', 'the', 'first', '.']


Removing whitespaces or not ; 
When developing a simple tokenizer, whether we should encode whitespaces as separate character or just remove them depend on our application and its requirements. removing whitespaces reduce memory usage and computation requirement. However keeping whitespaces can be useful if we train model that are sensitive to structure of text(python code sensitive to indentation and spacing). In this case we are going to remove the whitespaces.

The tokenization scheme that we had devised above works well on simple sample text. Modify it so that it also handle punctuation marks like question mark, quotation marks, and the double dashes etc. 

In [12]:
text="Hello, world. Is this-- a test?"
result=re.split(r'([,.:;?!_"()\']|--|\s)',text)
result=[item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


Now we got a basic tokenizer working, apply to edith wharton the verdict

In [14]:
preprocessed=re.split(r'([,.:;?!_"()\']|--|\s)',raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:100])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']


In [15]:
print(len(preprocessed))

4690


# Step 2 -> Create Token IDs

Create a list of all unique tokens and sort them alphabetically to determine the vocab size

In [21]:
all_words=set(preprocessed)
all_words=sorted(all_words)
vocab_size=len(all_words)
# for i in range(vocab_size):
#     print(all_words[i])
print(vocab_size)

1130


Create the vocab

In [24]:
vocab={token:integer for integer,token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
    print(item)
    if i>=100:
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)
('How', 52)
('I', 53)
('If', 54)
('In', 55)
('It', 56)
('Jack', 57)
('Jove', 58)
('Just', 59)
('Lord', 60)
('Made', 61)
('Miss', 62)
('Money', 63)
('Monte', 64)
('Moon-dancers', 65)
('Mr', 66)
('Mrs', 67)
('My', 68)
('Never', 69)
('No', 70)
('Now', 71)
('Nutley', 72)
('Of', 73)
('Oh', 74)
('On', 75)
('Once', 76)
('Only', 77)
('

Implement a complete tokenizer in Python.
Class will have an encode method that splits text into tokens and carries out the string to integer mapping 
In addition we will implement a decode method that carries out the reverse integer-to-string mapping to convert token id to word

In [29]:
class SimpleToeknizer:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.:;?!_"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        ids=[self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        # replace the spaces before the specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [32]:
tokenizer=SimpleToeknizer(vocab)
text=""""It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)  
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [33]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

Now lets apply it to something not in vocab

In [34]:
text='Hello, do you like tea?'
print(tokenizer.encode(text))

KeyError: 'Hello'

# Adding special contxt tokens
In prev sections, we implemented a simpletokenizer and  applied it to a passage from training set.
Now, we will modify this tokenizer to handle unknown words 


In [36]:
all_tokens=sorted(set(preprocessed))
all_tokens.extend(['<|endoftext|>','<|unk|>'])
vocab={token:integer for integer,token in enumerate(all_tokens)}

In [37]:
len(vocab)

1132

In [40]:
#printing last 5 entries of updated vocab
for i,items in enumerate(list(vocab.items())[-5::]):
    print(items)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [41]:
class simpletokenizerv2:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}
    
    def encode(self,text):
        preprocessed=re.split(r'([,.:;?!_"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        preprocessed=[
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids=[self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        # replace the spaces before the specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [43]:
tokenizer=simpletokenizerv2(vocab)
text1="Hello, do you like tea?"
text2="In the sunlit terraces of the palace."
text=" <|endoftext|> ".join([text1,text2])
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [44]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [45]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

# Byte pair encoding
We have implmented a simple tokenization scheme.
Now we will go to a more sophisticated scheme based on concept of byte pair encoding.
BPE tokenizer is used in LLMs like GPT-2, GPT-3 and the original model used in ChatGPT.
 

Since implementing BPE is relatively complicated, we will use python open source library called tiktoken(used by Openai).
This lib implemented BPE algo very efficiently based on source code in Rust.



In [2]:
! pip install tiktoken

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (45 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 2.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.6/804.6 kB 2.9 MB/s  0:00:00 eta 0:00:01
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [tiktoken]2/7 [idna]


In [4]:
import importlib
import tiktoken
print('tiktoken version:', importlib.metadata.version('tiktoken'))

tiktoken version: 0.14.0


In [5]:
tokenizer=tiktoken.get_encoding('gpt2')

In [6]:
text=(
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)
integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [9]:
strings=tokenizer.decode(integers)
print(strings)

,.?  the you do In like someofaces sun terr teaHellolitPlaceunknown<|endoftext|>


Lets take another example to illustrate how the BPE tokenizer deals with unknown tokens

In [10]:
integers=tokenizer.encode("Ajhjdaf ierds")
print(integers)

[32, 73, 71, 73, 67, 1878, 220, 959, 9310]


In [11]:
strings=tokenizer.decode(integers)
print(strings)

Ajhjdaf ierds


# Creating input-target pairs
Now we implement a data loader that fetches the input-target pairs using sliding window approach


In [3]:
import tiktoken

tokenizer=tiktoken.get_encoding('gpt2')
with open("the-verdict.txt",'r',encoding='utf-8') as f:
    raw_text=f.read()
enc_text=tokenizer.encode(raw_text)
print(int(len(enc_text)))

5145


Now we remove first 50 tokens for demonstration for a interesting text passage

In [4]:
enc_sample=enc_text[50:]


In [5]:
context_size=4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


Processing the input along with the targets, which are inputs shifted by one position, we can then create the next word prediction tasks as :

In [6]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(context,"--->",desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


Everything on left of arrow refers to an input for LLM, the token id on right represent the target that LLM supposed to predict 

For illustration convert token ids into text


In [8]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(tokenizer.decode(context),"--->",tokenizer.decode([desired]))   

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


# Implementing a Data Loader
For efficient data loader implementation, we will use pytorch's builtin dataset and dataloader class

Step 1: Tokenize the entire text  
Step 2: Use a sliding window to chunk the book into overlapping sequence of max_length  
Step 3: Return the total number of rows in the dataset  
Step 4: Return a single row from dataset  

In [9]:
! pip install torch

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.0 MB/s  0:03:50m0:00:0100:05m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 5.9 MB/s  0:01:52m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.9 MB/s  0:00:24m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 5.5 MB/s  0:00:40m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 7.1 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 7.2 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 6.9 MB/s  0:01:00m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/1

In [1]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        # Tokenize the entire text
        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

        #Use a sliding window to chunk the book into overlapping sequence of max_length
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]   
            target_chunk=token_ids[i+1:i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, index):
        return self.input_ids[index],self.target_ids[index]

Now code will use GPTdatasetV1 to load the inputs in batches via a pytorch dataloader
 

Steps   
1. initialize tokenizer
2. create dataset
3. drop_last= True drops the last batch if it is shorter than specified batch_size to prevent loss spiking during training
4. The number of CPU processes to use for preprocessing

In [2]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,
                         stride=128,shuffle=True,drop_last=True,
                         num_workers=0):
    #initialize the tokenizer
    tokenizer=tiktoken.get_encoding("gpt2")

    #create dataset
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)

    # create a bootloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [3]:
with open("the-verdict.txt",'r',encoding='utf-8') as f:
    raw_text=f.read()


In [26]:
import torch 
print("pytorch version:", torch.__version__)
dataloader=create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False
)
data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

pytorch version: 2.14.0+cu130
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [27]:
second_batch=next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [29]:
dataloader=create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False,
    num_workers=2
)
data_iter=iter(dataloader)
inputs,targets=next(data_iter)
print("inputs :\n",inputs)
print("targets :\n",targets)

inputs :
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets :
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


# Creating token embeddings


In [1]:
import torch
input_ids=torch.tensor([2,3,5,1])

For simplicity, suppose we have a small vocab of 6 words and we want to create embedding of size 3   
6 words are -> quick fox is in the house

In [2]:
vocab_size=6
output_dim=3
torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(vocab_size,output_dim)


In [3]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [4]:
print(embedding_layer(torch.tensor([1])))

tensor([[0.9178, 1.5810, 1.3010]], grad_fn=<EmbeddingBackward0>)


In [5]:
print(embedding_layer(torch.tensor(input_ids)))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


/tmp/ipykernel_6279/3846183200.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  print(embedding_layer(torch.tensor(input_ids)))


# Positional Embedding


Previously we focused on small embedding sizes.  
We now consider more realistic and useful embedding size and encode the input tokens into 256 dim vector representation.  
This is smaller than original gpt 3.
we assume that token ids are created by bpe tokenizer that we implemented earlier which has vocab size of 50257 

In [6]:
import importlib
import tiktoken

In [7]:
tokenizer=tiktoken.get_encoding('gpt2')

In [8]:
import torch
vocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

Using the token_embedding_layer above, if we sample data from data loader, we embed each token in each batch into a 256 dim vector. if thee we have a size of 8 with four tokens each, the result will be an 8x4x256 tensor

In [9]:
max_length=4
dataloader=create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=max_length,shuffle=False)
data_iter=iter(dataloader)
inputs,targets=next(data_iter)

In [10]:
print("Token ids : \n",inputs)
print('\nInputs shape: \n',inputs.shape)

Token ids : 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape: 
 torch.Size([8, 4])


Lets now use the embedding layer to embed these token ids into 256 dim vectors 

In [11]:
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


For gpt model absolute embedding approach, we just need to create another embedding layer that has the same dimension

In [12]:
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)

In [13]:
pos_embeddings=pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [14]:
input_embeddings=token_embeddings+pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
